# CircumferenceRegressor Training

Trains the MLP that maps extracted 2D body widths → real 3D circumferences (neck, waist, hip, thigh, calf, wrist).

**Prerequisites:**
- `synthetic-pose/` on Drive (front + side photos + `measurements/body_*.json`)
- Trained YOLO-pose weights (`runs/pose/best.pt` on Drive)
- Trained YOLO-seg weights (`runs/seg/best.pt` on Drive)

**Setup:** Runtime > Change runtime type > **GPU (T4)**

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics rich opencv-python

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")

## 2. Mount Drive + Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
set -e
rm -rf /content/pointsx_repo
git clone -q https://github.com/feed7362/PointsX.git /content/pointsx_repo
cd /content/pointsx_repo && pip install -q -e .
echo "PointsX installed."

## 3. Configuration

In [ ]:
# ============================================================
# CONFIGURE: paths on Drive
# ============================================================
DRIVE_ROOT      = "/content/drive/MyDrive/PointsX"
DRIVE_POSE_DS   = f"{DRIVE_ROOT}/synthetic-pose"          # front+side photos + measurements/
DRIVE_POSE_MODEL = f"{DRIVE_ROOT}/runs/pose/best.pt"
DRIVE_SEG_MODEL  = f"{DRIVE_ROOT}/runs/seg/best.pt"

# Local SSD copies (faster I/O during inference over thousands of images)
LOCAL_POSE_DS   = "/content/synthetic-pose"
LOCAL_POSE_MODEL = "/content/yolo11n-pose.pt"
LOCAL_SEG_MODEL  = "/content/yolo11n-seg.pt"
FEATURES_NPZ    = "/content/regression_features.npz"
OUTPUT_MODEL    = "/content/circumference_regressor.pt"

import os, shutil
os.makedirs(LOCAL_POSE_DS, exist_ok=True)

## 4. Copy Dataset + Models to Local Storage

In [ ]:
# Dataset: need train/images, val/images, and measurements/
if not os.path.isdir(f"{LOCAL_POSE_DS}/measurements"):
    print("Copying pose dataset from Drive (this may take a few minutes)...")
    !cp -r "{DRIVE_POSE_DS}" "{LOCAL_POSE_DS}"
else:
    print("Local dataset already present.")

# Models
if not os.path.exists(LOCAL_POSE_MODEL):
    shutil.copy2(DRIVE_POSE_MODEL, LOCAL_POSE_MODEL)
if not os.path.exists(LOCAL_SEG_MODEL):
    shutil.copy2(DRIVE_SEG_MODEL, LOCAL_SEG_MODEL)

print("\nSanity check:")
for p in (LOCAL_POSE_MODEL, LOCAL_SEG_MODEL):
    print(f"  {p}: {os.path.getsize(p) / 1e6:.1f} MB")
for split in ("train", "val"):
    n = len(os.listdir(f"{LOCAL_POSE_DS}/{split}/images"))
    print(f"  {split}/images: {n} files")
n_meas = len(os.listdir(f"{LOCAL_POSE_DS}/measurements"))
print(f"  measurements/:    {n_meas} JSONs")

## 5. Build Regression Training Set

For every body: runs pose + seg on front/side photos → extracts 2D widths → pairs with ground-truth circumferences from the measurements JSON. Output: `regression_features.npz` with `features (N,28)` and `targets (N,6)`.

In [ ]:
!python -m pointsx.regression.build_dataset \
    --pose-root  {LOCAL_POSE_DS} \
    --pose-model {LOCAL_POSE_MODEL} \
    --seg-model  {LOCAL_SEG_MODEL} \
    --output     {FEATURES_NPZ} \
    --device     cuda

In [ ]:
# Peek at the produced tensors
import numpy as np
d = np.load(FEATURES_NPZ)
feats, targets = d["features"], d["targets"]
print(f"features: {feats.shape}   dtype={feats.dtype}")
print(f"targets:  {targets.shape} dtype={targets.dtype}")
print("\nFeature stats (column mean ± std, first 10):")
for i in range(10):
    print(f"  f[{i:02d}]: {feats[:, i].mean():7.2f} ± {feats[:, i].std():6.2f}")
print("\nTarget stats (circumference, cm):")
for i, name in enumerate(["neck", "waist", "hip", "thigh", "calf", "wrist"]):
    print(f"  {name:<6}: {targets[:, i].mean():6.1f} ± {targets[:, i].std():5.1f}  "
          f"range [{targets[:, i].min():5.1f}, {targets[:, i].max():5.1f}]")

## 6. Train CircumferenceRegressor

In [ ]:
!python -m pointsx.regression.train \
    --data    {FEATURES_NPZ} \
    --output  {OUTPUT_MODEL} \
    --epochs  300 \
    --lr      1e-3 \
    --batch-size 64

## 7. Evaluate on Held-Out Split

In [ ]:
import torch, numpy as np
from pointsx.regression.model import CircumferenceRegressor

# Reload the .npz and do a deterministic 80/20 val split mirroring train.py
d = np.load(FEATURES_NPZ)
feats = torch.tensor(d["features"], dtype=torch.float32)
targets = torch.tensor(d["targets"], dtype=torch.float32)

torch.manual_seed(42)
n = len(feats)
perm = torch.randperm(n)
val_idx = perm[: int(n * 0.2)]
val_f, val_t = feats[val_idx], targets[val_idx]

model = CircumferenceRegressor()
model.load_state_dict(torch.load(OUTPUT_MODEL, map_location="cpu", weights_only=True))
model.eval()

with torch.no_grad():
    preds = model(val_f).numpy()

gt = val_t.numpy()
errs = preds - gt
print("Per-circumference error on the held-out split:")
print(f"  {'':<7}  {'MAE':>6}  {'RMSE':>6}  {'err<2cm':>8}")
for i, name in enumerate(["neck", "waist", "hip", "thigh", "calf", "wrist"]):
    mae  = np.mean(np.abs(errs[:, i]))
    rmse = np.sqrt(np.mean(errs[:, i] ** 2))
    pct  = np.mean(np.abs(errs[:, i]) < 2.0) * 100
    print(f"  {name:<7}  {mae:5.2f}   {rmse:5.2f}   {pct:6.1f}%")

In [ ]:
# Scatter plot: predicted vs. ground-truth for each circumference
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, i, name in zip(axes.flat, range(6),
                        ["neck", "waist", "hip", "thigh", "calf", "wrist"]):
    ax.scatter(gt[:, i], preds[:, i], alpha=0.4, s=12)
    lo = min(gt[:, i].min(), preds[:, i].min())
    hi = max(gt[:, i].max(), preds[:, i].max())
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1, alpha=0.7)
    ax.set_xlabel("Ground-truth (cm)")
    ax.set_ylabel("Predicted (cm)")
    ax.set_title(f"{name}  MAE={np.mean(np.abs(preds[:, i] - gt[:, i])):.2f} cm")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Save to Drive

In [ ]:
import shutil
dst = f"{DRIVE_ROOT}/models/circumference_regressor.pt"
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copy2(OUTPUT_MODEL, dst)
print(f"Saved: {dst}")

# Also save the feature .npz in case you want to retrain later
shutil.copy2(FEATURES_NPZ, f"{DRIVE_ROOT}/regression_features.npz")
print(f"Saved: {DRIVE_ROOT}/regression_features.npz")

---
## ✅ Full stack ready

On Drive you now have:
- `runs/pose/best.pt` — keypoint detector
- `runs/seg/best.pt` — silhouette detector
- `models/circumference_regressor.pt` — 2D widths → 3D circumferences

Use them together via `MeasurementPipeline`:
```python
from pointsx.pipeline import MeasurementPipeline
pipe = MeasurementPipeline(
    pose_model_path="runs/pose/best.pt",
    seg_model_path="runs/seg/best.pt",
    regression_model_path="models/circumference_regressor.pt",
)
result = pipe("front.jpg", "side.jpg", height_cm=175.0)
print(result.to_dict())
```